In [0]:
from pyspark.sql import Row, functions as F, types as T
from delta.tables import DeltaTable
from datetime import datetime, timezone
import requests
import json
import time
import logging
import uuid

In [0]:
# ---------------------------------------------------------------------------
# Logging Configuration
# ---------------------------------------------------------------------------
NOTEBOOK_NAME    = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split("/")[-1]

log              = logging.getLogger(NOTEBOOK_NAME)
log.setLevel(logging.INFO)
if not log.handlers:
    _handler     = logging.StreamHandler()
    _handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    )
    log.addHandler(_handler)

# ---------------------------------------------------------------------------
# Dynamic Configuration
# ---------------------------------------------------------------------------
dbutils.widgets.dropdown("env", "dev", ["dev", "tst", "prd"], "Environment")
env         = dbutils.widgets.get("env")
CATALOG          = f"{env}_corpdatav2"
BRONZE_DB        = f"{CATALOG}.bronze"
SILVER_DB        = f"{CATALOG}.silver"
AUDIT_CATALOG    = f"{env}_audit"
LOGS             = f"{AUDIT_CATALOG}.logs"

LOGS_TABLE       = f"{LOGS}.{NOTEBOOK_NAME}"
TARGET_BRONZE    = f"{BRONZE_DB}.ADX_API_UserAccount"
TARGET_SILVER    = f"{SILVER_DB}.ADX_API_UserAccount"

BASE_APIM        = "https://apim.api-dev.seaspancorp.com"
ADEXES_APIM      = f"{BASE_APIM}/adaxes/useractivity"

APIKEY           = dbutils.secrets.get(scope="azure-scope", key="adb-adx-apim")

HEADERS          = {
    "Content-Type": "application/json",
    "apiKey": APIKEY
}

PARAMS = {
    "filter":"((objectCategory=person)(objectClass=user))",
    "properties": "givenName,sn,mail,mobile,employeeID,whenCreated,objectTypeCode,lastLogon"
}

SEARCH_BODY      = {
    "criteria": {
        "objectTypes": [
            {
                "type": "user",
                "items": {
                    "type": 1,
                    "items": [],
                    "logicalOperator": 1
                }
            }
        ]
    },
    "select": {
        "properties": "givenName,sn,mail,mobile,adm-customattributetext39,employeeID,employeeNumber,whenCreated,objectTypeCode,lastLogon"
    }
}

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
MAX_RETRIES      = 3
BATCH_RUN_ID     = str(uuid.uuid4())

log.info(f"Environment: {env}")
log.info(f"Batch Run ID: {BATCH_RUN_ID}")
log.info(f"Catalog: {CATALOG}")
log.info(f"Max Retries: {MAX_RETRIES}")

In [0]:
audit_schema = T.StructType([
    T.StructField("EventID", T.StringType(), True),
    T.StructField("batch_run_id", T.StringType(), True),
    T.StructField("api_type", T.StringType(), True),
    T.StructField("http_method", T.StringType(), True),
    T.StructField("api_url", T.StringType(), True),
    T.StructField("input_value", T.StringType(), True),
    T.StructField("request_body", T.StringType(), True),
    T.StructField("attempt_number", T.IntegerType(), True),
    T.StructField("request_timestamp", T.TimestampType(), True),
    T.StructField("response_code", T.IntegerType(), True),
    T.StructField("response_body", T.StringType(), True),
    T.StructField("duration_ms", T.LongType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("execution_context", T.StringType(), True),
])

bronze_schema = T.StructType([
    T.StructField("Raw_data", T.StringType(), True),
    T.StructField("Api_metadata", T.StringType(), True),
    T.StructField("_is_normalized", T.BooleanType(), True),
    T.StructField("_ingestion_ts", T.TimestampType(), True),
])

silver_schema = T.StructType([
    T.StructField("dn", T.StringType(), True),
    T.StructField("firstName", T.StringType(), True),
    T.StructField("lastName", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("SFUserId", T.StringType(), True),
    T.StructField("lastLogon", T.StringType(), True),
    T.StructField("whenCreated", T.StringType(), True),
    T.StructField("isDisabled", T.BooleanType(), True),
    T.StructField("expirationDate", T.StringType(), True),
    T.StructField("LoadDateUTC", T.TimestampType(), True),
])

In [0]:
# ---------------------------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------------------------
def call_api(url, method="POST", headers=None, body=None, params=None, api_type="adx", input_value=None):
    """Pure API caller with retry logic for single-hit endpoints.
    Returns result dict with all attempts attached.
    Raises RuntimeError on HTTP 400-428 after max retries.
    """
    attempts = []
    last_status_code = None

    for attempt in range(1, MAX_RETRIES + 1):
        event_id = str(uuid.uuid4())
        start_ts = time.time()
        request_ts = datetime.now()

        try:
            response = requests.request(method, url, headers=headers, json=body, params=params, timeout=60)
            duration_ms = int((time.time() - start_ts) * 1000)
            last_status_code = response.status_code

            if 200 <= response.status_code <= 204:
                attempts.append({
                    "EventID": event_id,
                    "batch_run_id": BATCH_RUN_ID,
                    "api_type": api_type,
                    "http_method": method,
                    "api_url": url,
                    "input_value": input_value,
                    "request_body": json.dumps(body) if body else None,
                    "attempt_number": attempt,
                    "request_timestamp": request_ts,
                    "response_code": response.status_code,
                    "response_body": response.text[:4000],
                    "duration_ms": duration_ms,
                    "status": "SUCCESS",
                    "error_message": None,
                    "execution_context": None,
                })
                return {
                    "success": True,
                    "response_data": response.json() if response.text else None,
                    "api_metadata": {
                        "Status": "SUCCESS",
                        "ErrorMessage": None,
                        "HttpStatus": response.status_code,
                        "Timestamp": request_ts.strftime('%Y-%m-%dT%H:%M:%S'),
                    },
                    "attempts": attempts,
                }
            else:
                status = "RETRY" if attempt < MAX_RETRIES else "FAILED"
                attempts.append({
                    "EventID": event_id,
                    "batch_run_id": BATCH_RUN_ID,
                    "api_type": api_type,
                    "http_method": method,
                    "api_url": url,
                    "input_value": input_value,
                    "request_body": json.dumps(body) if body else None,
                    "attempt_number": attempt,
                    "request_timestamp": request_ts,
                    "response_code": response.status_code,
                    "response_body": response.text[:4000],
                    "duration_ms": duration_ms,
                    "status": status,
                    "error_message": f"HTTP {response.status_code}",
                    "execution_context": None,
                })
                if attempt < MAX_RETRIES:
                    time.sleep(2 ** attempt)

        except requests.exceptions.RequestException as exc:
            duration_ms = int((time.time() - start_ts) * 1000)
            status = "RETRY" if attempt < MAX_RETRIES else "FAILED"
            attempts.append({
                "EventID": event_id,
                "batch_run_id": BATCH_RUN_ID,
                "api_type": api_type,
                "http_method": method,
                "api_url": url,
                "input_value": input_value,
                "request_body": json.dumps(body) if body else None,
                "attempt_number": attempt,
                "request_timestamp": request_ts,
                "response_code": None,
                "response_body": None,
                "duration_ms": duration_ms,
                "status": status,
                "error_message": str(exc),
                "execution_context": None,
            })
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)

    # Max retries exhausted
    if last_status_code and 400 <= last_status_code <= 428:
        raise RuntimeError(
            f"Non-retryable client error (HTTP {last_status_code}) after {MAX_RETRIES} attempts for {url}"
        )

    return {
        "success": False,
        "response_data": None,
        "api_metadata": {
            "Status": "FAILED",
            "ErrorMessage": f"Max retries ({MAX_RETRIES}) exhausted",
            "HttpStatus": last_status_code,
            "Timestamp": datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        },
        "attempts": attempts,
    }


def log_audit(result):
    """Write all attempt entries from a result to the audit logs table."""
    exec_context = json.dumps({
        "notebook_path": dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get(),
        "cluster_id": spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "unknown"),
        "env": env
    })

    for entry in result["attempts"]:
        entry["execution_context"] = exec_context

    if not result["attempts"]:
        log.warning("No audit entries to write.")
        return

    df = spark.createDataFrame(result["attempts"], schema=audit_schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(LOGS_TABLE)
    log.info(f"Wrote {len(result['attempts'])} audit entries to {LOGS_TABLE}")


def write_to_bronze(result):
    """Append row to bronze table. SUCCESS → _is_normalized=False, FAILED/empty → _is_normalized=True."""
    is_normalized = not (result["success"] and result["response_data"])

    row = Row(
        Raw_data=json.dumps(result["response_data"]) if result["response_data"] else None,
        Api_metadata=json.dumps(result["api_metadata"]),
        _is_normalized=is_normalized,
        _ingestion_ts=datetime.now(timezone.utc),
    )

    df = spark.createDataFrame([row], schema=bronze_schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TARGET_BRONZE)
    log.info(f"Appended 1 row to {TARGET_BRONZE} (_is_normalized={is_normalized})")
    display(df)

In [0]:
# ---------------------------------------------------------------------------
# Normalize Bronze → Silver
# ---------------------------------------------------------------------------
def normalize_to_silver():
    """Read un-normalized bronze rows, parse JSON, apply Talend-equivalent transformations, and overwrite silver table."""

    bronze_df = spark.table(TARGET_BRONZE).filter(F.col("_is_normalized") == False)

    if not bronze_df.head(1):
        log.warning("No un-normalized rows in bronze. Skipping.")
        return

    # Parse Raw_data JSON — response is a list of user objects
    user_schema = T.ArrayType(T.StructType([
        T.StructField("dn", T.StringType(), True),
        T.StructField("accountStatus", T.StructType([
            T.StructField("isDisabled", T.BooleanType(), True),
            T.StructField("expirationDate", T.StringType(), True),
        ]), True),
        T.StructField("properties", T.StructType([
            T.StructField("givenName", T.ArrayType(T.StringType()), True),
            T.StructField("sn", T.ArrayType(T.StringType()), True),
            T.StructField("mail", T.ArrayType(T.StringType()), True),
            T.StructField("employeeID", T.ArrayType(T.StringType()), True),
            T.StructField("lastLogon", T.ArrayType(T.StringType()), True),
            T.StructField("whenCreated", T.ArrayType(T.StringType()), True),
        ]), True),
    ]))

    parsed_df = (
        bronze_df
        .withColumn("users", F.from_json(F.col("Raw_data"), user_schema))
        .filter(F.col("users").isNotNull())
        .withColumn("user", F.explode(F.col("users")))
    )

    # Filter: remove records where employeeID is null or empty
    # parsed_df = parsed_df.filter(
    #     (F.col("user.properties.employeeID")[0].isNotNull())
    #     & (F.trim(F.col("user.properties.employeeID")[0]) != "")
    # )

    # Default timestamp for null/empty/unspecified date fields
    DEFAULT_TS = F.lit("1900-01-01 00:00:00.0000000")

    # Flatten with Talend-equivalent transformations
    silver_df = parsed_df.select(
        F.col("user.dn").alias("dn"),
        F.col("user.properties.givenName")[0].alias("firstName"),
        F.col("user.properties.sn")[0].alias("lastName"),
        F.col("user.properties.mail")[0].alias("email"),
        # SFUserId mapped from employeeID
        F.coalesce(F.regexp_replace(F.col("user.properties.employeeID")[0], r"\s+", ""), F.lit("")).alias("SFUserId"),
        # lastLogon: null/empty/unspecified → 1900-01-01, else parse timestamp
        F.when(
            (F.col("user.properties.lastLogon")[0].isNull())
            | (F.trim(F.col("user.properties.lastLogon")[0]) == "")
            | (F.lower(F.col("user.properties.lastLogon")[0]) == "unspecified"),
            DEFAULT_TS
        ).otherwise(
            F.concat(F.date_format(F.to_timestamp(F.col("user.properties.lastLogon")[0]), "yyyy-MM-dd HH:mm:ss"), F.lit(".0000000"))
        ).alias("lastLogon"),
        # whenCreated: null/empty/unspecified → 1900-01-01, else parse timestamp
        F.when(
            (F.col("user.properties.whenCreated")[0].isNull())
            | (F.trim(F.col("user.properties.whenCreated")[0]) == "")
            | (F.lower(F.col("user.properties.whenCreated")[0]) == "unspecified"),
            DEFAULT_TS
        ).otherwise(
            F.concat(F.date_format(F.to_timestamp(F.col("user.properties.whenCreated")[0]), "yyyy-MM-dd HH:mm:ss"), F.lit(".0000000"))
        ).alias("whenCreated"),
        F.col("user.accountStatus.isDisabled").alias("isDisabled"),
        # expirationDate: null/empty/unspecified → 1900-01-01, else parse and subtract 1 minute
        F.when(
            (F.col("user.accountStatus.expirationDate").isNull())
            | (F.trim(F.col("user.accountStatus.expirationDate")) == "")
            | (F.lower(F.col("user.accountStatus.expirationDate")) == "unspecified"),
            DEFAULT_TS
        ).otherwise(
            F.concat(F.date_format(F.from_utc_timestamp(F.to_timestamp(F.col("user.accountStatus.expirationDate")), "America/Los_Angeles") - F.expr("INTERVAL 1 MINUTE"), "yyyy-MM-dd HH:mm:ss"), F.lit(".0000000"))
        ).alias("expirationDate"),
        F.current_timestamp().alias("LoadDateUTC"),
    ).distinct()

    # Write to silver (overwrite)
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(TARGET_SILVER)

    log.info(f"Silver table written → {TARGET_SILVER}")

    # Update _is_normalized flag in bronze table for processed rows
    DeltaTable.forName(spark, TARGET_BRONZE).update(
        condition=F.col("_is_normalized") == False,
        set={"_is_normalized": F.lit(True)}
    )
    log.info(f"Bronze table updated: _is_normalized set to True for processed rows in {TARGET_BRONZE}")

In [0]:
# ---------------------------------------------------------------------------
# Main Execution
# ---------------------------------------------------------------------------
if __name__ == "__main__":

    # --- Step 3: Fetch User Accounts ---
    log.info("Step 1: Fetching user accounts from Adaxes APIM...")
    result = call_api(ADEXES_APIM, method="GET", headers=HEADERS, body=SEARCH_BODY, params=PARAMS, api_type="adx")
    log_audit(result)

    if not result["success"]:
        raise RuntimeError(f"API call FAILED: {result['api_metadata']['ErrorMessage']}")

    # --- Step 4: Write to Bronze ---
    log.info("Step 4: Writing to bronze...")
    write_to_bronze(result)

    # --- Step 5: Normalize to Silver ---
    log.info("Step 5: Normalizing bronze to silver...")
    normalize_to_silver()

    log.info(f"Pipeline completed successfully. Batch Run ID: {BATCH_RUN_ID}")